In [ ]:
import pandas as pd

In [ ]:
# read enrichment data
rich = pd.read_csv('', dtype={'id_number' : str,
                                             'cellphone_number' : str})

# rich['umuzi_email'] = rich['umuzi_email'].str.strip().str.lower()

rich = (
    rich
    .sort_values('learner_id')
    .drop_duplicates(subset='umuzi_email', keep='last')
)

In [ ]:
# read indicator 5 file
five = pd.read_csv(
    '',
    dtype={
            'id_number' : 'str',
            'cellphone_number' : 'str'
                   }
)
five.shape


In [ ]:
# read the data from current indicator 7 file
current = pd.read_csv('',
                      dtype={
                       'id_number' : 'str',
                       'cellphone_number' : 'str'
                   })
current.shape

In [ ]:
# read previous indicator 7 file
previous = pd.read_csv(
    '',
    dtype={
                       'id_number' : 'str',
                       'cellphone_number' : 'str'
                   }
)
previous.shape

In [ ]:
# read database indicator 7 file
database = pd.read_csv(
    '',
    dtype={
            'id_number' : 'str',
            'cellphone_number' : 'str'
                   }
)
database.shape

In [ ]:
# concatenate the two dataframes
combined = pd.concat([previous, current, database], ignore_index=True)

In [ ]:
combined.shape

In [ ]:
# check for duplicates
combined.duplicated().sum()

In [ ]:
# remove the duplicates
combined.drop_duplicates(keep='first', inplace=True)

In [ ]:
combined.shape

In [ ]:
excluded_indices = combined.loc[(combined['first_name'].isna()) & (combined['last_name'].isna())].index

enriched_records = (
    combined.loc[excluded_indices]
    [['umuzi_email', 'service_used', 'service_name', 'date_service_accessed', 'age_service_accessed', 'month_of_service_accessed', 'age_range']]
    .merge(
        rich.drop(columns=['email']), on='umuzi_email', how='left'
    )

)

In [ ]:
# drop said records from combined
combined.drop(labels=excluded_indices, axis=0, inplace=True)

In [ ]:
combined.shape

In [ ]:
enriched_records.shape

In [ ]:
excluded_indices.__len__()

In [ ]:
#remerge the enriched records
combined = (
    pd.concat([combined, enriched_records[combined.columns]])
    .sort_values(by='umuzi_email')
    .reset_index(drop=True)
)

In [ ]:
combined.head()

In [ ]:
# New version for participants list
participantsCols = [
    'first_name', 'last_name', 'id_number', 'date_service_accessed', 'service_name',
    'cellphone_number', 'age_service_accessed', 'nearest_metro', 'province', 'race', 'gender'
]

participantsList = combined[participantsCols].copy()

In [ ]:
participantsList.head()

In [ ]:
participantsList.loc[participantsList['first_name'].isna()]

In [ ]:
set(combined.loc[participantsList.loc[participantsList['first_name'].isna()].index, 'umuzi_email'])

In [ ]:
participantsList.loc[participantsList['first_name'].isna()].index.__len__()

In [ ]:
combined.loc[343, "first_name"] == participantsList.loc[343, "first_name"]

In [ ]:
ind6 = (combined
 .groupby(by=['umuzi_email'], as_index=False)
 .agg({
        "service_used" :"first" ,
        "service_name": lambda x: list(set(x)),  # collect unique services
        # keep personal info with "first" (since it’s the same across rows)
        "learner_id": "first",
        "application_id": "first",
        "application_date": "first",
        "first_name": "first",
        "last_name": "first",
        "cellphone_number": "first",
        "id_number": "first",
        "race": "first",
        "gender": "first",
        "date_of_birth": "first",
        'age_service_accessed': "first",
        'age_range': "first",
        "has_disability_or_differently_abled": "first",
        "nearest_metro": "first",
        "province": "first",
        "residential_area_type": "first",
        "date_service_accessed" : "max",
        'month_of_service_accessed': "max"
    }))

In [ ]:
ind6 = ind6[[
    'learner_id', 'application_id', 'umuzi_email',
    'first_name', 'last_name', 'cellphone_number',
    'id_number', 'gender', 'date_of_birth', 'age_service_accessed', 'age_range', 'race',
    'residential_area_type', 'has_disability_or_differently_abled',
    'application_date', 'nearest_metro', 'province', 'date_service_accessed',
    'month_of_service_accessed', 'service_used', 'service_name'
]]

ind6.head()

In [ ]:
ind6.to_csv('../Sink Datasets/indicator_6_data.csv', index=False)

In [ ]:
# write to excel
with pd.ExcelWriter('../Sink Datasets/IDC Report Consolidated.xlsx') as writer:
    five.to_excel(writer, sheet_name='Indicator 5', index=False)
    ind6.to_excel(writer, sheet_name='Indicator 6', index=False)
    combined.to_excel(writer, sheet_name='Indicator 7', index=False)
    participantsList.to_excel(writer, sheet_name='Support Paricipants List', index=False)